<a href="https://colab.research.google.com/github/Jeisson893/Hello-World/blob/main/Dash_desercion_academica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# DASHBOARD INTERACTIVO DE DESERCIÓN ACADÉMICA (v2 storytelling)  (pegar en una celda nueva
# al final del notebook de limpieza; usa el df ya limpio o el _LIMPIO.csv)
# Genera un HTML autocontenido (sin librerías externas), lo muestra en
# Colab y lo deja en /content/dashboard_desercion.html
# =====================================================================
import pandas as pd, numpy as np, json, re
from IPython.display import HTML, display

# ---------- 1. Cargar datos limpios ----------
import glob
import glob
RUTA_XLSX = "/content/DESERCION_ACADEMICA_LIMPIO.xlsx"   # <- archivo subido a Colab

def a_bool(serie):   # Excel puede guardar TRUE/FALSE como texto; esto lo normaliza
    if serie.dtype == bool: return serie
    return serie.astype(str).str.strip().str.upper().isin(["TRUE", "1", "VERDADERO", "SI", "SÍ"])

if "df" in globals() and "PERIODO_COMPLETO" in df.columns:
    d = df.copy()                                   # usa el df del notebook si existe
else:
    ruta_x = RUTA_XLSX if glob.glob(RUTA_XLSX) else (glob.glob("/content/DESERCION_ACADEMICA_LIMPIO*.xlsx") or [RUTA_XLSX])[0]
    d = pd.read_excel(ruta_x, engine="openpyxl")    # hoja 1 del Excel
d["PERIODO_COMPLETO"] = a_bool(d["PERIODO_COMPLETO"])

# El pregrado solo se analiza en periodos completos (el posgrado usa todos)
excluidos = int(((d["TIPO_FORMACION"] == "PREGRADO") & (~d["PERIODO_COMPLETO"])).sum())
d = d[~((d["TIPO_FORMACION"] == "PREGRADO") & (~d["PERIODO_COMPLETO"]))].copy()
SIN_PREGRADO = ["2020-1", "2020-2", "2021-2"]   # igual que PERIODOS_SIN_PREGRADO

# ---------- 2. Etiquetas legibles ----------
ACENTOS = {"administracion":"administración","ingenieria":"ingeniería","tecnologia":"tecnología",
 "tecnica":"técnica","tecnico":"técnico","educacion":"educación","economia":"economía",
 "contaduria":"contaduría","gestion":"gestión","informacion":"información","produccion":"producción",
 "comunicacion":"comunicación","construccion":"construcción","biologia":"biología","quimica":"química",
 "fisica":"física","psicologia":"psicología","pedagogia":"pedagogía","filosofia":"filosofía",
 "matematicas":"matemáticas","tecnologo":"tecnólogo","especializacion":"especialización",
 "maestria":"maestría","bogota":"Bogotá","boyaca":"Boyacá","atlantico":"Atlántico","cordoba":"Córdoba",
 "narino":"Nariño","quindio":"Quindío","choco":"Chocó","bolivar":"Bolívar","guainia":"Guainía",
 "vaupes":"Vaupés","caqueta":"Caquetá","sistemas":"sistemas","electronica":"electrónica",
 "mecanica":"mecánica","electrica":"eléctrica","geologia":"geología","metalurgica":"metalúrgica",
 "ambiental":"ambiental","agroindustriales":"agroindustriales","regencia":"regencia","salud":"salud"}
def bonito(s):
    if not isinstance(s, str): return s
    t = s.strip().lower()
    t = re.sub(r"[a-záéíóúñ]+", lambda m: ACENTOS.get(m.group(0), m.group(0)), t)
    t = t.replace("d.c.", "D.C.")
    return t[:1].upper() + t[1:]

def causal(fila):
    t = str(fila.get("NOMBRE_ESTADO", "")).upper()
    m = re.search(r"LIT\.?\s*([A-Z])", t)
    if not m: return None
    if "130" in t: n = "Pregrado"            # Acuerdo 130/98 art. 80
    elif re.search(r"\b0?52\b|POSGRADO", t): n = "Posgrado"   # Acuerdo 052/12 art. 15
    else: n = fila["TIPO_FORMACION"].title()   # respaldo
    return f"{n} - Lit. {m.group(1)}"

NIVEL = {"TECNICO PROFESIONAL":"Técnico profesional","TECNOLOGO":"Tecnólogo","PROFESIONAL":"Profesional",
         "ESPECIALIZACION":"Especialización","MAESTRIA":"Maestría","DOCTORADO":"Doctorado"}
GEDAD = {"20 O MENOS":"20 o menos","21-25":"21 a 25","26-30":"26 a 30","31-40":"31 a 40","MAS DE 40":"Más de 40"}

out = pd.DataFrame({
    "p":  d["PERIODO_CLAVE"],
    "tf": d["TIPO_FORMACION"].str.title(),
    "nv": d["NIVEL_FORMACION"].map(NIVEL),
    "fa": d["FACULTAD_ACADEMICA"].map(bonito),
    "mo": d["MODALIDAD"].str.title(),
    "se": d["SEDE_AGRUPADA"].map(bonito),
    "ge": d["GENERO"].astype(str).str.upper().str[0],
    "ed": d["EDAD"].astype(int),
    "ga": d["GRUPO_EDAD"].astype(str).map(GEDAD),
    "es": pd.to_numeric(d["ESTRATO"], errors="coerce"),
    "pr": d["PROGRAMA_AGRUPADO"].map(bonito),
    "dp": d["DEPTO_ORIGEN"].map(bonito),
    "mc": a_bool(d["MISMA_CIUDAD_SEDE"]).astype(int),
    "ca": d.apply(causal, axis=1),
})
out["es"] = out["es"].where(out["es"].notna(), None)
out = out.astype(object).where(out.notna(), None)
payload = {"rows": out.to_dict("records"),
           "per": sorted(d["PERIODO_CLAVE"].unique().tolist()),
           "sin": SIN_PREGRADO, "exc": excluidos}

# ---------- 3. Plantilla HTML (CSS + JS) ----------
HTML_T = r"""<!DOCTYPE html><html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>Deserción académica</title>
<style>
*{box-sizing:border-box;-webkit-print-color-adjust:exact;print-color-adjust:exact}
:root{--ink:#1b2a41;--mut:#6b7785;--bg:#f2f5f9;--ln:#dde3ea;--dk:#1f3a5f;--bs:#6f93c4;--ac:#e8743b;--tl:#1a9e8f}
body{margin:0;background:var(--bg);color:var(--ink);font:14px/1.45 Inter,"Segoe UI",system-ui,sans-serif}
.w{max-width:1360px;margin:0 auto;padding:0 28px 50px}
.hd{background:var(--dk);color:#fff;margin:0 -28px;padding:26px 28px 22px}.hd h1{margin:0;font-size:26px;font-weight:700;letter-spacing:-.3px}
.hd p{margin:6px 0 0;color:#cfdcf0;font-size:15px;max-width:1000px}.hd b{color:#fff}.hd small{display:block;margin-top:8px;color:#9fb4d3;font-size:12px}
.fl{position:sticky;top:0;z-index:9;background:#fff;margin:0 -28px;padding:10px 28px;border-bottom:1px solid var(--ln);display:flex;flex-wrap:wrap;gap:10px 14px;align-items:flex-end;box-shadow:0 2px 6px rgba(0,0,0,.05)}
.fl label{display:flex;flex-direction:column;font-size:11px;color:var(--mut);gap:3px;text-transform:uppercase;letter-spacing:.4px}
select,button{font:inherit;font-size:13px;padding:5px 8px;border:1px solid #c6cfda;border-radius:6px;background:#fff;color:var(--ink)}button{cursor:pointer}
.sg{display:inline-flex}.sg button{border-radius:0;margin-left:-1px}.sg button:first-child{border-radius:6px 0 0 6px;margin:0}.sg button:last-child{border-radius:0 6px 6px 0}.sg button.on,.tg button.on{background:var(--dk);color:#fff;border-color:var(--dk)}
.rs{background:var(--ac);color:#fff;border-color:var(--ac);font-weight:600}.cnt{margin-left:auto;font-size:12px;color:var(--mut)}
.chs{display:flex;gap:8px;flex-wrap:wrap;margin:10px 0 0;min-height:4px}.chip{background:#fff3ec;border:1px solid var(--ac);color:#a8480f;border-radius:14px;padding:2px 10px;font-size:12px;cursor:pointer}
.g{display:grid;grid-template-columns:repeat(12,1fr);gap:14px;margin-top:14px}.s3{grid-column:span 3}.s4{grid-column:span 4}.s5{grid-column:span 5}.s6{grid-column:span 6}.s7{grid-column:span 7}.s8{grid-column:span 8}.s12{grid-column:span 12}
@media(max-width:1000px){.g>*{grid-column:span 12!important}.k6{grid-template-columns:1fr 1fr!important}}
.c{background:#fff;border:1px solid var(--ln);border-radius:10px;padding:14px 16px;min-width:0;break-inside:avoid}.c h3{margin:0 0 2px;font-size:14px;font-weight:700}.n{font-size:12px;color:var(--mut);margin:0 0 8px}
.k6{display:grid;grid-template-columns:repeat(6,1fr);gap:14px;margin-top:16px}.k{border-top:3px solid var(--bs)}.k:first-child{border-top-color:var(--ac)}.kl{font-size:11px;color:var(--mut);text-transform:uppercase;letter-spacing:.4px}
.k b{display:block;font-size:28px;font-weight:700;margin:2px 0;color:var(--dk);letter-spacing:-.5px}.k small{color:var(--mut);font-size:12px}
.sec{display:flex;align-items:baseline;gap:10px;margin:30px 0 0;border-bottom:2px solid var(--dk);padding-bottom:4px}.sec i{background:var(--dk);color:#fff;border-radius:50%;width:24px;height:24px;display:inline-grid;place-items:center;font-style:normal;font-size:13px;font-weight:700}
.sec h2{margin:0;font-size:19px}.sec span{color:var(--mut);font-size:12px}.ins{margin:8px 0 0;padding:8px 12px;background:#fff3ec;border-left:4px solid var(--ac);border-radius:0 6px 6px 0;font-size:13.5px}.ins b{color:#a8480f}
.br{display:grid;grid-template-columns:minmax(80px,1.15fr) 1.3fr auto;gap:8px;align-items:center;margin:3px 0;padding:2px 4px;border-radius:5px;font-size:12.5px}.x{cursor:pointer}.x:hover{background:#eef3fa}.br.on{background:#fff3ec}.br.dim{opacity:.45}
.t{height:13px;background:#e8edf3;border-radius:4px;overflow:hidden}.t i{display:block;height:100%;border-radius:4px}.v{text-align:right;white-space:nowrap;font-size:12px;color:var(--mut)}.v b{color:var(--ink)}
.st{display:flex;height:24px;border-radius:5px;overflow:hidden;background:#e8edf3}.st i{display:block;height:100%}.lg{display:flex;flex-wrap:wrap;gap:4px 14px;font-size:12px;color:#444;margin-top:8px}
.lg span:before{content:"";display:inline-block;width:10px;height:10px;border-radius:3px;background:var(--c);margin-right:5px}
.tg{float:right}.tg button{border-radius:0;padding:3px 10px;font-size:12px}
.hm{border-collapse:separate;border-spacing:3px;width:100%;font-size:12px}.hm th{font-weight:500;color:var(--mut);font-size:11px}.hm td{text-align:center;border-radius:5px;padding:8px 4px;min-width:42px}.hm th.r{text-align:left;white-space:nowrap}
.sc{overflow-x:auto}svg text{font:11px Inter,system-ui,sans-serif;fill:#5b6775}.ft{margin-top:26px;display:grid;grid-template-columns:repeat(3,1fr);gap:14px;font-size:12px;color:var(--mut)}.ft b{color:var(--ink)}
@media print{.fl{position:static;box-shadow:none}.rs,.pb{display:none}.g{gap:10px}.sec{break-after:avoid}}@page{size:A4 landscape;margin:8mm}
</style></head><body><div class="w">
<div class="hd"><h1>Deserción académica por pérdida de cupo</h1><p id="sub"></p><small>Cifras = número de deserciones (no tasas, no hay matriculados). Haz clic en cualquier barra para filtrar todo el tablero.</small></div>
<div class="fl" id="fl"></div><div class="chs" id="ch"></div>
<div class="k6" id="kp"></div>

<div class="sec"><i>1</i><h2>¿Cuándo ocurre?</h2><span>Evolución por periodo</span></div><p class="ins" id="i1"></p>
<div class="g"><div class="c s7"><h3>Pregrado por periodo</h3><p class="n">Solo periodos completos; «s/d» = sin datos de pregrado. Barra oscura = pico.</p><div id="c_pre"></div></div>
<div class="c s5"><h3>Posgrado por periodo</h3><p class="n">Escala propia. Las cifras bajas desde 2024-1 pueden ser del registro: confirmar con la oficina.</p><div id="c_pos"></div></div>
<div class="c s12"><div class="tg" data-k="mod"><button data-v="n" class="on">Número</button><button data-v="p">% del periodo</button></div><h3>Modalidad en el tiempo</h3><p class="n">Periodos completos. ¿Se está moviendo la deserción hacia lo virtual?</p><div id="c_mod"></div></div></div>

<div class="sec"><i>2</i><h2>¿Dónde se concentra?</h2><span>Programas, unidades y sedes</span></div><p class="ins" id="i2"></p>
<div class="g"><div class="c s5"><h3>Top 15 programas</h3><p class="n" id="n_pr"></p><div id="b_pr"></div></div>
<div class="s7" style="display:grid;gap:14px;align-content:start"><div class="c"><h3>Facultad / unidad académica</h3><p class="n">«Estudios a Distancia» y «Propio de Seccional» son unidades, no facultades.</p><div id="b_fa"></div></div>
<div class="g" style="margin:0"><div class="c s6"><h3>Nivel de formación</h3><div id="b_nv"></div></div><div class="c s6"><h3>Sede</h3><div id="b_se"></div></div></div></div></div>

<div class="sec"><i>3</i><h2>¿Quién es el estudiante?</h2><span>Perfil sociodemográfico</span></div><p class="ins" id="i3"></p>
<div class="g"><div class="c s4"><h3>Grupo de edad</h3><div id="b_ga"></div></div>
<div class="c s4"><h3>Género</h3><div id="b_ge"></div><h3 style="margin-top:18px">¿Estudiaba en su ciudad?</h3><div id="b_mc"></div></div>
<div class="c s4"><h3>Estrato</h3><p class="n" id="n_es"></p><div id="b_es"></div></div>
<div class="c s12"><h3>Departamento de origen (top 10)</h3><p class="n" id="n_dp"></p><div id="b_dp"></div></div></div>

<div class="sec"><i>4</i><h2>¿Por qué pierden el cupo?</h2><span>Causal registrada</span></div><p class="ins" id="i4"></p>
<div class="g"><div class="c s4"><h3>Distribución de causales</h3><p class="n">Pregrado: Acuerdo 130/1998 art. 80. Posgrado: Acuerdo 052/2012 art. 15. El archivo no incluye el texto de cada literal.</p><div id="b_ca"></div></div>
<div class="c s4"><h3>Causal por nivel de formación</h3><p class="n">Cada barra suma 100% de ese nivel.</p><div id="s_nv"></div></div>
<div class="c s4"><h3>Causal por modalidad</h3><p class="n">Cada barra suma 100% de esa modalidad.</p><div id="s_mo"></div></div>
<div class="s12 lg" id="lg_ca" style="margin:0"></div></div>

<div class="sec"><i>5</i><h2>Cruces clave</h2><span>Cuanto más oscuro, mayor participación</span></div>
<div class="g"><div class="c s7"><div class="tg" data-k="h1"><button data-v="n" class="on">Número</button><button data-v="p">% fila</button></div><h3>Nivel de formación × causal</h3><div class="sc" id="h1"></div></div>
<div class="c s5"><div class="tg" data-k="h2"><button data-v="n" class="on">Número</button><button data-v="p">% fila</button></div><h3>Grupo de edad × modalidad</h3><div class="sc" id="h2"></div></div></div>
<div class="ft" id="ft"></div><p style="text-align:right"><button class="pb" onclick="window.print()">Imprimir / guardar PDF</button></p>
</div><script>
const DATA=__DATA__,R=DATA.rows,PER=DATA.per,SIN=DATA.sin;
const CO=['Pregrado - Lit. A','Pregrado - Lit. B','Pregrado - Lit. C','Pregrado - Lit. D','Pregrado - Lit. E','Posgrado - Lit. A','Posgrado - Lit. B','Posgrado - Lit. C'];
const CC=['#a9c2e3','#6f93c4','#1f3a5f','#4aa3a2','#9ac9c0','#e8743b','#f2a65a','#c0392b'];
const NIV=['Técnico profesional','Tecnólogo','Profesional','Especialización','Maestría','Doctorado'],GA=['20 o menos','21 a 25','26 a 30','31 a 40','Más de 40'],MOD=['Presencial','Virtual','Distancia'];
const BS='#6f93c4',DK='#1f3a5f',AC='#e8743b',KEYS=['tf','fa','mo','se','ge','nv','ca','pr','dp','ga'],CHIP=['nv','ca','pr','dp','ga'];
const $=i=>document.getElementById(i),fmt=n=>n.toLocaleString('es-CO'),pc=(a,b)=>b?(a/b*100).toFixed(1).replace('.',',')+'%':'–',pl=p=>'S'+p.slice(-1)+' '+p.slice(0,4);
const F0={d:PER[0],h:PER[PER.length-1],tf:'',fa:'',mo:'',se:'',ge:'',nv:'',ca:'',pr:'',dp:'',ga:''};let F={...F0};const T={mod:'n',h1:'n',h2:'n'};
const uniq=k=>[...new Set(R.map(r=>r[k]).filter(Boolean))].sort((a,b)=>a.localeCompare(b,'es'));
const cnt=(rows,k)=>{const m={};rows.forEach(r=>{const v=typeof k=='function'?k(r):r[k];if(v!=null)m[v]=(m[v]||0)+1});return m};
const srt=m=>Object.entries(m).sort((a,b)=>b[1]-a[1]),med=a=>{if(!a.length)return 0;a=[...a].sort((x,y)=>x-y);const m=a.length>>1;return a.length%2?a[m]:(a[m-1]+a[m])/2};
const RF=skip=>R.filter(r=>r.p>=F.d&&r.p<=F.h&&KEYS.every(k=>k==skip||!F[k]||r[k]==F[k]));
// ---- filtros (solo los más relevantes + clic en gráficos)
const sel=(k,l,o,all)=>`<label>${l}<select data-k="${k}">${all?`<option value="">${all}</option>`:''}${o.map(v=>`<option>${v}</option>`).join('')}</select></label>`;
const seg=(k,l,o)=>`<label>${l}<span class="sg" data-k="${k}">${o.map(([v,t])=>`<button data-v="${v}">${t}</button>`).join('')}</span></label>`;
$('fl').innerHTML=sel('d','Desde',PER)+sel('h','Hasta',PER)+seg('tf','Formación',[['','Todos'],['Pregrado','Pregrado'],['Posgrado','Posgrado']])+sel('fa','Facultad',uniq('fa'),'Todas')+sel('se','Sede',uniq('se'),'Todas')+sel('mo','Modalidad',MOD,'Todas')+seg('ge','Género',[['','Todos'],['M','Hombres'],['F','Mujeres']])+'<button class="rs" id="rs">Restablecer</button><span class="cnt" id="ct"></span>';
function sync(){document.querySelectorAll('#fl select').forEach(s=>s.value=F[s.dataset.k]);document.querySelectorAll('#fl .sg').forEach(g=>g.querySelectorAll('button').forEach(b=>b.classList.toggle('on',b.dataset.v==F[g.dataset.k])));
 $('ch').innerHTML=CHIP.filter(k=>F[k]).map(k=>`<span class="chip x" data-k="${k}" data-v="${F[k]}">${F[k]} ✕</span>`).join('')}
document.addEventListener('change',e=>{const s=e.target.closest('#fl select');if(!s)return;F[s.dataset.k]=s.value;if(F.d>F.h){if(s.dataset.k=='d')F.h=F.d;else F.d=F.h}draw()});
document.addEventListener('click',e=>{let t;
 if(t=e.target.closest('#fl .sg button')){F[t.parentNode.dataset.k]=t.dataset.v;return draw()}
 if(e.target.id=='rs'){F={...F0};return draw()}
 if(t=e.target.closest('.tg button')){const g=t.parentNode;T[g.dataset.k]=t.dataset.v;g.querySelectorAll('button').forEach(b=>b.classList.toggle('on',b==t));return draw()}
 if(t=e.target.closest('.x')){const k=t.dataset.k,v=t.dataset.v;F[k]=F[k]==v?'':v;draw()}});
// ---- componentes
function bars(id,k,m,tot,o={}){let e=srt(m);if(o.order)e=o.order.filter(x=>m[x]!=null).map(x=>[x,m[x]]);if(o.top)e=e.slice(0,o.top);const mx=Math.max(1,...e.map(x=>x[1])),hs=k&&F[k];
 $(id).innerHTML=e.length?e.map(([l,n],i)=>{const on=hs&&F[k]==l,col=o.col?o.col(l):on?AC:(!hs&&i==0&&!o.order?DK:BS);
 return `<div class="br ${k?'x':''} ${on?'on':''} ${hs&&!on?'dim':''}" ${k?`data-k="${k}" data-v="${l}"`:''} title="${l}: ${fmt(n)}"><span>${l}</span><div class="t"><i style="width:${n/mx*100}%;background:${col}"></i></div><span class="v"><b>${fmt(n)}</b> · ${pc(n,tot)}</span></div>`}).join(''):'<p class="n">Sin datos con los filtros actuales.</p>'}
function stack(id,pairs,tot){$(id).innerHTML=tot?`<div class="st">${pairs.map(([l,n,c])=>`<i title="${l}: ${fmt(n)}" style="width:${n/tot*100}%;background:${c}"></i>`).join('')}</div><div class="lg">${pairs.map(([l,n,c])=>`<span style="--c:${c}"><b>${l}</b> ${fmt(n)} (${pc(n,tot)})</span>`).join('')}</div>`:''}
function cols(id,vals){const W=560,H=250,b=W/vals.length,mx=Math.max(1,...vals.map(v=>v.n||0)),pk=vals.reduce((a,v)=>v.n>(a.n||0)?v:a,{});
 $(id).innerHTML=`<svg viewBox="0 0 ${W} ${H}" width="100%">${vals.map((v,i)=>{const x=i*b+b*.12,w=b*.76,h=v.na?170:(v.n/mx)*165,y=205-h,c=v.na?'#eef1f5':v==pk?DK:BS;
 return `<g><title>${v.p}: ${v.na?'s/d':fmt(v.n)}</title><rect x="${x}" y="${y}" width="${w}" height="${h}" rx="4" fill="${c}"/><text x="${x+w/2}" y="${v.na?200:y-5}" text-anchor="middle" style="font-weight:700;fill:#1b2a41">${v.na?'s/d':v.n}</text><text x="${x+w/2}" y="222" text-anchor="middle">S${v.p.slice(-1)}</text>${v.p.endsWith('1')?`<text x="${x+w/2}" y="237" text-anchor="middle" style="font-weight:600">${v.p.slice(0,4)}</text>`:''}</g>`}).join('')}</svg>`}
function line(id,rows){const ps=PER.filter(p=>!SIN.includes(p)&&p>=F.d&&p<=F.h),W=900,H=290,L=40,Rr=110,C=[DK,'#1a9e8f',AC];
 const S=MOD.map(m=>ps.map(p=>{const a=rows.filter(r=>r.p==p),n=a.filter(r=>r.mo==m).length;return T.mod=='n'?n:(a.length?n/a.length*100:0)}));
 const mx=Math.max(1,...S.flat()),X=i=>L+(ps.length>1?i*(W-L-Rr)/(ps.length-1):0),Y=v=>240-v/mx*200;
 $(id).innerHTML=`<svg viewBox="0 0 ${W} ${H}" width="100%">${[0,.25,.5,.75,1].map(f=>`<line x1="${L}" x2="${W-Rr}" y1="${Y(mx*f)}" y2="${Y(mx*f)}" stroke="#e8edf3"/><text x="${L-6}" y="${Y(mx*f)+4}" text-anchor="end">${Math.round(mx*f)}${T.mod=='p'?'%':''}</text>`).join('')}
 ${S.map((s,j)=>`<polyline fill="none" stroke="${C[j]}" stroke-width="2.5" points="${s.map((v,i)=>X(i)+','+Y(v)).join(' ')}"/>${s.map((v,i)=>`<circle cx="${X(i)}" cy="${Y(v)}" r="3.5" fill="${C[j]}"><title>${MOD[j]} ${pl(ps[i])}: ${T.mod=='n'?v:v.toFixed(1)+'%'}</title></circle>`).join('')}<text x="${W-Rr+8}" y="${Y(s[s.length-1]||0)+4}" style="fill:${C[j]};font-weight:700">${MOD[j]}</text>`).join('')}
 ${ps.map((p,i)=>`<text x="${X(i)}" y="260" text-anchor="middle">S${p.slice(-1)}</text>${p.endsWith('1')?`<text x="${X(i)}" y="274" text-anchor="middle" style="font-weight:600">${p.slice(0,4)}</text>`:''}`).join('')}</svg>`}
function heat(id,rows,rk,ro,ck,co,mode,tot){const m={};rows.forEach(r=>{if(r[rk]&&r[ck]){const k=r[rk]+'|'+r[ck];m[k]=(m[k]||0)+1}});
 const cs=co.filter(c=>rows.some(r=>r[ck]==c)),rs=ro.filter(x=>rows.some(r=>r[rk]==x)),rt={};rs.forEach(x=>rt[x]=cs.reduce((s,c)=>s+(m[x+'|'+c]||0),0));
 const val=(x,c)=>{const n=m[x+'|'+c]||0;return mode=='n'?n:(rt[x]?n/rt[x]*100:0)},mx=Math.max(1,...rs.flatMap(x=>cs.map(c=>val(x,c))));
 $(id).innerHTML=`<table class="hm"><tr><th></th>${cs.map(c=>`<th>${c.replace(' - ','<br>')}</th>`).join('')}${tot?'<th>Total</th>':''}</tr>${rs.map(x=>`<tr><th class="r">${x}</th>${cs.map(c=>{const n=m[x+'|'+c]||0,v=val(x,c),a=n?.1+.8*v/mx:.04;return `<td title="${x} / ${c}: ${fmt(n)}" style="background:rgba(31,58,95,${a});color:${a>.5?'#fff':'#1b2a41'}">${n?(mode=='n'?fmt(n):v.toFixed(0)+'%'):'·'}</td>`}).join('')}${tot?`<td style="color:#6b7785;font-weight:600">${fmt(rt[x])}</td>`:''}</tr>`).join('')}</table>`}
const spark=v=>{const mx=Math.max(1,...v),mn=Math.min(...v),w=120,h=26;return v.length>1?`<svg viewBox="0 0 ${w} ${h}" width="120" height="26"><polyline fill="none" stroke="${AC}" stroke-width="2" points="${v.map((x,i)=>(i*w/(v.length-1))+','+(h-3-(x-mn)/Math.max(1,mx-mn)*(h-6))).join(' ')}"/></svg>`:''};
// ---- render
function draw(){sync();
 const rows=RF(),N=rows.length;$('ct').textContent=`Mostrando ${fmt(N)} de ${fmt(R.length)} deserciones`;
 const pre=rows.filter(r=>r.tf=='Pregrado').length,pos=N-pre,cac=cnt(rows,'ca'),top=srt(cac)[0],muj=rows.filter(r=>r.ge=='F').length,tr=rows.filter(r=>r.mc==0).length,eds=rows.map(r=>r.ed);
 const tp=PER.filter(p=>!SIN.includes(p)&&p>=F.d&&p<=F.h).map(p=>[p,rows.filter(r=>r.p==p).length]);
 $('sub').innerHTML=N?`<b>${fmt(N)}</b> deserciones entre ${F.d} y ${F.h}: <b>${pc(pre,N)}</b> pregrado y <b>${pc(pos,N)}</b> posgrado.${top?` Causal principal: <b>${top[0]}</b> (${pc(top[1],N)}).`:''}`:'No hay registros con los filtros actuales.';
 $('kp').innerHTML=[['Total deserciones',fmt(N),spark(tp.map(x=>x[1]))],['Pregrado / posgrado',`${pc(pre,N)} / ${pc(pos,N)}`,`${fmt(pre)} · ${fmt(pos)} registros`],['Edad mediana',(N?med(eds):0)+' años',N?`Rango ${Math.min(...eds)}–${Math.max(...eds)}`:''],['Mujeres',pc(muj,N),`${fmt(muj)} de ${fmt(N)}`],['Se trasladó de ciudad',pc(tr,N),`${fmt(tr)} estudiaban fuera de su ciudad`],['Causal principal',top?top[0].replace('Pregrado - ','Pre · ').replace('Posgrado - ','Pos · '):'–',top?`${fmt(top[1])} · ${pc(top[1],N)}`:'']].map(([a,b,c])=>`<div class="c k"><span class="kl">${a}</span><b ${a=='Pregrado / posgrado'?'style="font-size:22px"':''}>${b}</b><small>${c}</small></div>`).join('');
 // 1 cuándo
 const per=tf=>PER.filter(p=>p>=F.d&&p<=F.h).map(p=>({p,n:rows.filter(r=>r.p==p&&r.tf==tf).length,na:tf=='Pregrado'&&SIN.includes(p)}));
 cols('c_pre',per('Pregrado'));cols('c_pos',per('Posgrado'));line('c_mod',rows);
 if(tp.length>1){const pk=tp.reduce((a,x)=>x[1]>a[1]?x:a),l=tp[tp.length-1],pv=tp[tp.length-2],dl=pv[1]?(l[1]-pv[1])/pv[1]*100:0;$('i1').innerHTML=`El pico fue en <b>${pl(pk[0])}</b> (${fmt(pk[1])} deserciones). El último periodo (${pl(l[0])}) registró <b>${fmt(l[1])}</b>, ${dl>=0?'+':''}${dl.toFixed(0)}% frente al periodo anterior.`}else $('i1').textContent='Amplía el rango de periodos para ver la tendencia.';
 // 2 dónde
 const rp=RF('pr'),pr=cnt(rp.filter(r=>r.pr&&r.pr!='Otros programas'),'pr');$('n_pr').textContent=`Excluye «Otros programas» (${fmt(rp.filter(r=>r.pr=='Otros programas').length)} casos de programas pequeños).`;
 bars('b_pr','pr',pr,rp.length,{top:15});
 const rfa=RF('fa'),rnv=RF('nv'),rse=RF('se'),se=cnt(rse,'se'),otr=se['Otras sedes'];delete se['Otras sedes'];if(otr)se['Otras sedes']=otr;
 bars('b_fa','fa',cnt(rfa,'fa'),rfa.length);bars('b_nv','nv',cnt(rnv,'nv'),rnv.length,{order:NIV});bars('b_se','se',se,rse.length);
 const t5=srt(cnt(rows.filter(r=>r.pr!='Otros programas'),'pr')).slice(0,5).reduce((s,x)=>s+x[1],0),tfa=srt(cnt(rows,'fa'))[0],tse=srt(cnt(rows,'se'))[0];
 $('i2').innerHTML=N?`Los 5 programas principales concentran <b>${pc(t5,N)}</b> de las deserciones. <b>${tfa[0]}</b> es la unidad con más casos (${pc(tfa[1],N)}) y la sede <b>${tse[0]}</b> aporta ${pc(tse[1],N)}.`:'';
 // 3 quién
 const rga=RF('ga');bars('b_ga','ga',cnt(rga,'ga'),rga.length,{order:GA});
 stack('b_ge',[['Hombres',N-muj,DK],['Mujeres',muj,AC]],N);stack('b_mc',[['En su ciudad',N-tr,DK],['Se trasladó',tr,AC]],N);
 const es=rows.filter(r=>r.es!=null),ne=N-es.length,em=cnt(es,r=>'Estrato '+r.es),ep=es.length?es.reduce((s,r)=>s+r.es,0)/es.length:0;
 $('n_es').textContent=`Promedio ${ep.toFixed(2).replace('.',',')} (excluye ${fmt(ne)} sin información). Sin matriculados por estrato no se puede hablar de mayor riesgo.`;
 const emo=srt(em)[0];em['Sin información']=ne;bars('b_es',null,em,N,{order:['Estrato 1','Estrato 2','Estrato 3','Estrato 4','Estrato 5','Estrato 6','Sin información']});
 const rdp=RF('dp'),dr=rdp.filter(r=>r.dp&&!['Ambiguo','No encontrado'].includes(r.dp));$('n_dp').textContent=`Excluye ${fmt(rdp.length-dr.length)} registros con departamento ambiguo o no encontrado.`;bars('b_dp','dp',cnt(dr,'dp'),rdp.length,{top:10});
 const tga=srt(cnt(rows,'ga'))[0];$('i3').innerHTML=N?`El perfil más frecuente: <b>${tga[0]} años</b>, ${muj>N/2?'mujer':'hombre'} (${pc(Math.max(muj,N-muj),N)}), ${emo?`<b>${emo[0].toLowerCase()}</b>, `:''}y <b>${pc(tr,N)}</b> se trasladó de ciudad para estudiar.`:'';
 // 4 por qué
 const rca=RF('ca');bars('b_ca','ca',cnt(rca,'ca'),rca.length,{order:CO,col:l=>CC[CO.indexOf(l)]});
 const sb=(id,key,order)=>{const rr=RF(key);$(id).innerHTML=order.filter(o=>rr.some(r=>r[key]==o)).map(o=>{const a=rr.filter(r=>r[key]==o),c=cnt(a,'ca'),t=CO.reduce((s,k)=>s+(c[k]||0),0)||1,on=F[key]==o;
  return `<div class="br x ${on?'on':''} ${F[key]&&!on?'dim':''}" data-k="${key}" data-v="${o}" style="grid-template-columns:92px 1fr auto"><span>${o}</span><div class="st">${CO.map((k,i)=>c[k]?`<i title="${k}: ${c[k]} (${pc(c[k],t)})" style="width:${c[k]/t*100}%;background:${CC[i]}"></i>`:'').join('')}</div><span class="v">n=${fmt(a.length)}</span></div>`}).join('')||'<p class="n">Sin datos.</p>'};
 sb('s_nv','nv',NIV);sb('s_mo','mo',MOD);$('lg_ca').innerHTML=CO.map((k,i)=>`<span style="--c:${CC[i]}">${k}</span>`).join('');
 const s2=srt(cac);$('i4').innerHTML=s2.length>1?`<b>${s2[0][0]}</b> explica ${pc(s2[0][1],N)} de las deserciones; junto con <b>${s2[1][0]}</b> suman ${pc(s2[0][1]+s2[1][1],N)}. Los literales de posgrado (en naranja) casi solo aparecen en especialización, maestría y doctorado.`:'';
 // 5 cruces
 heat('h1',rows,'nv',NIV,'ca',CO,T.h1,false);heat('h2',rows,'ga',GA,'mo',MOD,T.h2,true);
 $('ft').innerHTML=`<div><b>Cobertura.</b> ${SIN.join(', ')} no tienen datos de pregrado: el pregrado solo cuenta en periodos completos (${DATA.exc} registros excluidos). El posgrado usa todos los periodos.</div><div><b>Qué se mide.</b> Número de deserciones y % del total filtrado. Ninguna cifra es una tasa de deserción.</div><div><b>Cómo leer.</b> Un grupo con más deserciones puede ser simplemente un grupo más grande. Son asociaciones, no causas.</div>`}
draw();
</script></body></html>"""

html = HTML_T.replace("__DATA__", json.dumps(payload, ensure_ascii=False, default=str))
ARCHIVO = "/content/dashboard_desercion.html"
with open(ARCHIVO, "w", encoding="utf-8") as f:
    f.write(html)
print(f"✔ Dashboard generado: {ARCHIVO}  ({len(html)/1024:.0f} KB, {len(out)} registros)")

display(HTML(html))                      # vista interactiva dentro de Colab
try:                                     # descarga para abrirlo en el navegador
    from google.colab import files
    files.download(ARCHIVO)
except Exception:
    pass


✔ Dashboard generado: /content/dashboard_desercion.html  (905 KB, 3363 registros)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>